In [0]:
cleaned_order_df =  spark.read.parquet("/Volumes/workspace/default/superstore/data/silver/orders_cleaned", inferSchema = True, header = True)
display(cleaned_order_df)

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

dim_customer = cleaned_order_df.select(
    "customer_name",
    "segment",
    "market",
    "region",
    "country",
    "state"
).dropDuplicates().withColumn(
    "customer_id",
    monotonically_increasing_id()  #creating surrogate key
).select(
    "customer_id",
    "customer_name",
    "segment",
    "market",
    "region",
    "country",
    "state"
)

dim_product = cleaned_order_df.select(
    "product_name",
    "category",
    "sub_category"
).dropDuplicates().withColumn(
    "product_key",
    monotonically_increasing_id()    #creating surrogate key
).select(
    "product_key",
    "product_name",
    "category",
    "sub_category"
)

dim_region = cleaned_order_df.select(
    "market",
    "region",
    "country",
    "state"
).dropDuplicates().withColumn(
    "region_key",
    monotonically_increasing_id()     #creating surrogate key
).select(
    "region_key",
    "market",
    "region",
    "country",
    "state"
)

fact_sales = cleaned_order_df.join(
    dim_customer,
    on=[
        "customer_name",
        "segment",
        "market",
        "region",
        "country",
        "state"
    ],
    how="left"
).join(dim_product,
       on=[
    "product_name",
    "category",
    "sub_category"],
       how="left").join(dim_region,
       on=[
    "market",
    "region",
    "country",
    "state"],
       how="left").select(
    "order_id",
    "order_date",
    "ship_date",
    "customer_id",
    "product_key",
    "region_key",
    "sales",
    "state",
    "quantity",
    "discount",
    "profit",
    "shipping_cost"
)

In [0]:
%sql
DROP TABLE IF EXISTS fact_sales;
DROP TABLE IF EXISTS dim_customer;
DROP TABLE IF EXISTS dim_product;
DROP TABLE IF EXISTS dim_region;

In [0]:
fact_sales.write.mode("overwrite").saveAsTable("fact_sales")
dim_customer.write.mode("overwrite").saveAsTable("dim_customer")
dim_product.write.mode("overwrite").saveAsTable("dim_product")
dim_region.write.mode("overwrite").saveAsTable("dim_region")


In [0]:
#creating views for sql kpi creation

fact_sales.createOrReplaceTempView("fact_sales")
dim_customer.createOrReplaceTempView("dim_customer")
dim_product.createOrReplaceTempView("dim_product")
dim_region.createOrReplaceTempView("dim_region")

In [0]:
#writing to the gold layer

fact_sales.write.mode("overwrite").parquet("/Volumes/workspace/default/superstore/data/gold/fact_sales")
dim_customer.write.mode("overwrite").parquet("/Volumes/workspace/default/superstore/data/gold/dim_customer")
dim_product.write.mode("overwrite").parquet("/Volumes/workspace/default/superstore/data/gold/dim_product")
dim_region.write.mode("overwrite").parquet("/Volumes/workspace/default/superstore/data/gold/dim_region")